# StableBoundaryLayerGSPT SCM in Google Colab

This notebook installs Julia, enables IJulia, switches to a Julia runtime, and runs an optimized, zero-allocation SCM tendency kernel sanity check.


In [ ]:
%%bash
set -euo pipefail

JULIA_VERSION=1.10.4
JULIA_TAR="julia-${JULIA_VERSION}-linux-x86_64.tar.gz"
JULIA_URL="https://julialang-s3.julialang.org/bin/linux/x64/1.10/${JULIA_TAR}"

wget -q "${JULIA_URL}"
tar -xzf "${JULIA_TAR}"

# Colab generally allows this without sudo
cp -r "julia-${JULIA_VERSION}"/{bin,etc,include,lib,share} /usr/local/

julia -e 'using Pkg; Pkg.add("IJulia"); println("Installed Julia ", VERSION)'


## Switch Runtime to Julia

1. Refresh the browser page.
2. Open Runtime -> Change runtime type.
3. Select Julia 1.10.x if available.
4. Save and continue.

If Julia does not appear in the runtime list, stay on Python runtime and run Julia from shell using commands like !julia -e.


In [ ]:
using InteractiveUtils
println("Julia version: ", VERSION)


## Install Required Julia Packages

LinearAlgebra is part of the standard library, so only DifferentialEquations is added.


In [ ]:
using Pkg
Pkg.add(["DifferentialEquations", "BenchmarkTools"])
using LinearAlgebra
using DifferentialEquations
using BenchmarkTools
println("Packages ready.")


## Define SCM Kernel and Parameter Types

This contains our highly-optimized version featuring:
- Static workspace buffer extraction via `_get_face_diffusivity_buffers` to guarantee zero-allocation.
- Constant-time identity comparisons (`===`) for Symbols in the boundary conditions.
- Loop optimizations via `@inbounds` and `@simd` to unlock compiler vectorization.


In [ ]:
using LinearAlgebra

mutable struct SCMWorkspace{T}
    Km::Vector{T}
    Kh::Vector{T}
end

struct SCMParameters{T,W}
    N::Int
    dz::T
    z_centers::Vector{T}
    z_faces::Vector{T}
    f::T
    Ug::T
    Vg::T
    theta_a::T
    T_deep::T
    delta::T
    K_buoy::T
    beta::T
    l_0::T
    eta::T
    xi::T
    C_skin::T
    R_down::T
    lambda_s::T
    d_soil::T
    theta_top_bc::Symbol
    theta_top::T
    lambda_top::T
    workspace::W
end

function _get_face_diffusivity_buffers(p)
    # Directly accessing typed workspace fields avoids dynamic runtime checks
    if hasproperty(p, :workspace)
        ws = p.workspace
        return ws.Km, ws.Kh
    elseif hasproperty(p, :K_m_faces) && hasproperty(p, :K_h_faces)
        return p.K_m_faces, p.K_h_faces
    else
        error("Performance Error: Preallocated face buffers not found.")
    end
end

function scm_gspt_tendencies!(dX, X, p, t)
    N = p.N
    dz = p.dz
    z_centers = p.z_centers
    z_faces = p.z_faces

    f = p.f
    Ug = p.Ug
    Vg = p.Vg
    theta_a = p.theta_a
    T_deep = p.T_deep

    delta = p.delta
    K_buoy = p.K_buoy
    beta = p.beta
    l_0 = p.l_0
    eta = p.eta
    xi = p.xi
    kappa = 0.4
    Pr_t = 1.0

    C_skin = p.C_skin
    R_down = p.R_down
    sigma_SB = 5.67e-8
    rho_cp = 1200.0
    lambda_s = p.lambda_s
    d_soil = p.d_soil

    theta_top_bc = p.theta_top_bc
    theta_top_ref = p.theta_top
    lambda_top = p.lambda_top

    T_s = X[1]
    U = @view X[2:N+1]
    V = @view X[N+2:2N+1]
    theta = @view X[2N+2:3N+1]

    dU = @view dX[2:N+1]
    dV = @view dX[N+2:2N+1]
    dtheta = @view dX[2N+2:3N+1]

    K_m_faces, K_h_faces = _get_face_diffusivity_buffers(p)

    @inbounds @simd for i in 1:(N-1)
        dU_dz = (U[i+1] - U[i]) / dz
        dV_dz = (V[i+1] - V[i]) / dz
        dth_dz = (theta[i+1] - theta[i]) / dz

        z_face = z_faces[i+1]
        ell_z = (kappa * z_face) / (1.0 + (kappa * z_face) / l_0)

        stability_arg = clamp(beta * dth_dz * ell_z / theta_a, -40.0, 40.0)
        G_local = expm1(stability_arg)

        Delta_local = eta * (ell_z^2) * (dU_dz^2 + dV_dz^2) - K_buoy * G_local
        A = l_0 * Delta_local - delta
        e_star_xi = 0.5 * (A + hypot(A, xi))

        K_m_faces[i] = ell_z * sqrt(e_star_xi + delta)
        K_h_faces[i] = K_m_faces[i] / Pr_t
    end

    @inbounds for i in 2:(N-1)
        flux_U_top = K_m_faces[i] * (U[i+1] - U[i]) / dz
        flux_U_bot = K_m_faces[i-1] * (U[i] - U[i-1]) / dz

        flux_V_top = K_m_faces[i] * (V[i+1] - V[i]) / dz
        flux_V_bot = K_m_faces[i-1] * (V[i] - V[i-1]) / dz

        flux_H_top = K_h_faces[i] * (theta[i+1] - theta[i]) / dz
        flux_H_bot = K_h_faces[i-1] * (theta[i] - theta[i-1]) / dz

        dU[i] = f * (V[i] - Vg) + (flux_U_top - flux_U_bot) / dz
        dV[i] = -f * (U[i] - Ug) + (flux_V_top - flux_V_bot) / dz
        dtheta[i] = (flux_H_top - flux_H_bot) / dz
    end

    dU_dz_surf = (U[1] - 0.0) / dz
    dV_dz_surf = (V[1] - 0.0) / dz
    dth_dz_surf = (theta[1] - T_s) / dz

    ell_surf = (kappa * z_centers[1]) / (1.0 + (kappa * z_centers[1]) / l_0)
    stability_arg_surf = clamp(beta * dth_dz_surf * ell_surf / theta_a, -40.0, 40.0)
    G_surf = expm1(stability_arg_surf)
    Delta_surf = eta * (ell_surf^2) * (dU_dz_surf^2 + dV_dz_surf^2) - K_buoy * G_surf

    A_surf = l_0 * Delta_surf - delta
    e_star_surf = 0.5 * (A_surf + hypot(A_surf, xi))

    K_m_surf = ell_surf * sqrt(e_star_surf + delta)
    K_h_surf = K_m_surf / Pr_t

    flux_U_surf = K_m_surf * (U[1] - 0.0) / dz
    flux_V_surf = K_m_surf * (V[1] - 0.0) / dz
    flux_H_surf = K_h_surf * (theta[1] - T_s) / dz

    @inbounds begin
        flux_U_top1 = K_m_faces[1] * (U[2] - U[1]) / dz
        flux_V_top1 = K_m_faces[1] * (V[2] - V[1]) / dz
        flux_H_top1 = K_h_faces[1] * (theta[2] - theta[1]) / dz

        dU[1] = f * (V[1] - Vg) + (flux_U_top1 - flux_U_surf) / dz
        dV[1] = -f * (U[1] - Ug) + (flux_V_top1 - flux_V_surf) / dz
        dtheta[1] = (flux_H_top1 - flux_H_surf) / dz

        # Swift Symbol identity checks for zero allocation top BC handling
        flux_H_topN = if theta_top_bc === :dirichlet
            K_h_faces[N-1] * (theta_top_ref - theta[N]) / dz
        elseif theta_top_bc === :relaxation
            -lambda_top * (theta[N] - theta_top_ref)
        else
            0.0
        end

        dU[N] = f * (V[N] - Vg) + (0.0 - K_m_faces[N-1] * (U[N] - U[N-1]) / dz) / dz
        dV[N] = -f * (U[N] - Ug) + (0.0 - K_m_faces[N-1] * (V[N] - V[N-1]) / dz) / dz
        dtheta[N] = (flux_H_topN - K_h_faces[N-1] * (theta[N] - theta[N-1]) / dz) / dz

        R_net = R_down - sigma_SB * (T_s^4)
        H_sensible = rho_cp * flux_H_surf
        G_ground = lambda_s * (T_s - T_deep) / d_soil

        dX[1] = (1.0 / C_skin) * (R_net - H_sensible - G_ground)
    end
    return nothing
end

println("Kernel definitions loaded.")


## Initialize Sandbox Parameters and Run a Single RHS Evaluation


In [ ]:
N = 50
dz = 2.0
z_centers = collect(range(dz / 2, step = dz, length = N))
z_faces = collect(range(0.0, step = dz, length = N + 1))

# Preallocate face buffers for zero-allocation RHS behavior
ws = SCMWorkspace(zeros(N - 1), zeros(N - 1))

p = SCMParameters(
    N, dz, z_centers, z_faces,
    1e-4, 10.0, 0.0, 280.0, 275.0,
    1e-4, 1.0, 1.0, 100.0, 0.1, 0.01,
    1e4, 300.0, 1.0, 1.0,
    :neumann, 280.0, 0.0,
    ws
)

X = zeros(3N + 1)
X[1] = 278.0
X[2:N+1] .= 10.0
X[N+2:2N+1] .= 0.0
X[2N+2:3N+1] .= 280.0

dX = similar(X)
scm_gspt_tendencies!(dX, X, p, 0.0)

println("Success. dT_s/dt = ", dX[1])
println("Sample momentum tendency dU[1] = ", dX[2])
println("Sample thermal tendency dtheta[1] = ", dX[2N + 2])


## Optional Performance Check

This benchmark verifies fast repeated evaluations and confirms the workspace-buffer path is active.


In [ ]:
@btime scm_gspt_tendencies!($dX, $X, $p, 0.0)
